# Gemma AST Agent — Kaggle runner

**Setup (right panel → Settings):** Accelerator = **GPU T4 x2**, Internet = **On**.

**Inputs (right panel → Add Input):** the Gemma 4 model — search *Gemma 4*, choose framework **Transformers** and a small `-it` variant (E2B or E4B).

The code is downloaded from Google Drive in cell 1 (the zip must be shared as *Anyone with the link*). Alternatively, add a dataset containing `gemma-ast-agent.zip` as an input and set `DRIVE_ZIP_ID = ''`.

Then run cells top to bottom.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Find the code and the model in /kaggle/input

In [ ]:
import glob, os, shutil, urllib.request, zipfile

DRIVE_ZIP_ID = '1T-0A7mTLqNT8K2at-pxeCdNiAzRroYxr'  # Drive file id of gemma-ast-agent.zip ('' = use a Kaggle dataset input)
WORK = '/kaggle/working/gemma-ast-agent'
shutil.rmtree(WORK, ignore_errors=True)

if DRIVE_ZIP_ID:
    zpath = '/kaggle/working/gemma-ast-agent.zip'
    urllib.request.urlretrieve(f'https://drive.google.com/uc?export=download&id={DRIVE_ZIP_ID}', zpath)
    assert zipfile.is_zipfile(zpath), 'Download failed: check Internet is On and the zip is shared as Anyone with the link'
    zipfile.ZipFile(zpath).extractall('/kaggle/working')
else:
    # Kaggle usually unzips uploaded archives; handle both cases.
    src = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/ast_agent/__init__.py', recursive=True)]
    if src:
        shutil.copytree(os.path.dirname(src[0]), WORK)
    else:
        zips = glob.glob('/kaggle/input/**/gemma-ast-agent.zip', recursive=True)
        assert zips, 'Add your dataset with gemma-ast-agent.zip as an input'
        zipfile.ZipFile(zips[0]).extractall('/kaggle/working')
os.chdir(WORK)
print('code:', WORK, sorted(os.listdir('.')))

# Model: the folder that holds config.json (excluding our own repo).
cfgs = [p for p in glob.glob('/kaggle/input/**/config.json', recursive=True) if 'gemma-ast-agent' not in p]
assert cfgs, 'Add a Gemma 4 Transformers model as an input'
for c in cfgs: print('model candidate:', os.path.dirname(c))
MODEL_PATH = os.path.dirname(cfgs[0])
print('using:', MODEL_PATH)

## 2. Install and test (should say 15 passed)

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes pytest
!python -m pytest -q tests

## 3. Harness check with known fixes (every task should be RESOLVED)

In [ ]:
!python -m ast_agent.eval --tasks benchmarks/sample/tasks.jsonl --backend gold --conditions full --out /kaggle/working/results/gold 2>&1 | tail -12

## 4. Smoke test: 2 tasks, 1 condition
If you get CUDA out of memory, add `--load-in-4bit`.

In [ ]:
RESULTS = '/kaggle/working/results'
!python -m ast_agent.eval --tasks benchmarks/sample/tasks.jsonl --backend hf --model "$MODEL_PATH" \
  --dtype float16 --conditions ast_graph --limit 2 --max-new-tokens 1024 \
  --out "$RESULTS/smoke" 2>&1 | grep -v ' INFO '

In [ ]:
import json
r = json.loads(open(f'{RESULTS}/smoke/results.jsonl').readline())
print('resolved:', r['resolved'], '| apply errors:', r['attempts'][0]['apply_errors'])
print(r['attempts'][0]['response'][:3000])

## 5. Build your real benchmark (optional here; CPU is enough)
Clone a pure-Python library with a fast test suite at a fixed release tag, then plant bugs. Skip this cell to keep using the sample benchmark.

In [ ]:
LIB_URL = ''   # e.g. https://github.com/<owner>/<repo>.git
LIB_TAG = ''   # e.g. a release tag, for reproducibility
LIB_TEST_CMD = 'python -m pytest -q --tb=short -p no:cacheprovider'

if LIB_URL:
    name = LIB_URL.rstrip('/').split('/')[-1].removesuffix('.git')
    !git clone -q --depth 1 --branch $LIB_TAG $LIB_URL benchmarks/repos/$name
    !pip install -q -e benchmarks/repos/$name
    !python -m ast_agent.mutate --repo benchmarks/repos/$name --test-cmd "$LIB_TEST_CMD" --out benchmarks/$name --max-tasks 50 --seed 0
    TASKS = f'benchmarks/{name}/tasks.jsonl'
else:
    TASKS = 'benchmarks/sample/tasks.jsonl'
print('tasks:', TASKS)

## 6. Full experiment grid
For long runs use **Save Version → Save & Run All (Commit)** so it keeps running after you close the tab. Everything in `/kaggle/working/results` becomes the notebook's output.

In [ ]:
!python -m ast_agent.eval --tasks "$TASKS" --backend hf --model "$MODEL_PATH" \
  --dtype float16 --conditions file,ast_graph,ast_graph_sc,full,oracle --max-new-tokens 1024 \
  --out "$RESULTS/main" 2>&1 | grep -v ' INFO '

In [ ]:
from IPython.display import Markdown
Markdown(open(f'{RESULTS}/main/summary.md').read())